# 241 - Ward at every K, both feature sets

**Run 240, 241 and 242 in parallel** - three kernels, one per algorithm. They share the dataset cache read-only and write to separate run directories and separate sweep files, so they cannot collide.

**Before this:** `python rebuild_concat_cache.py --apply` must have finished. It is the only serial step.

**After all three:** run `243_cluster_statistics.ipynb`.

| | |
|---|---|
| method | Ward |
| feature sets | `concat_hg`, `concat_rawds` (gated cohort, one set of electrodes) |
| K | 5 .. 30 |
| cNMF fit iterations | 1000 final, 300 inside cross-validation |


In [1]:
import os, sys, json, subprocess, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
assert (ROOT / 'functions').exists(), f'run this from 02_FBM_Clustering, not {ROOT}'
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / 'functions'))
# cfg is lf_blob_clustering_config here - 02_FBM_Clustering has no config.py;
# that name belongs to 01_FBM_Analysis. RANDOM_STATE lives in this one.
import lf_concat as CC, lf_cluster_run as R, lf_runs as LR
import lf_blob_clustering_config as cfg

SCRIPT_NAME = '241_cluster_hierarchical.ipynb'
METHOD      = 'hierarchical'
METHOD_LBL  = 'Ward'
CACHE_DIR   = ROOT / 'outputs/_dataset/concat_source_v4'
FEATURE_SETS= ['concat_hg', 'concat_rawds']
K_RANGE     = [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]
N_ITER      = 1000          # convex NMF fit iterations (final fits)
RANDOM_STATE= cfg.RANDOM_STATE

INPUT_DIR = Path(r'\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\01_FBM_Analysis\outputs\04_ersp_LM_RAWONLY')
if not INPUT_DIR.exists():
    INPUT_DIR = (ROOT.parent / '01_FBM_Analysis' / 'outputs' / '04_ersp_LM_RAWONLY').resolve()

print('method      :', METHOD_LBL)
print('feature sets:', FEATURE_SETS)
print('K range     :', K_RANGE[0], '..', K_RANGE[-1], f'({len(K_RANGE)} values)')
print('cache       :', CACHE_DIR)

method      : Ward
feature sets: ['concat_hg', 'concat_rawds']
K range     : 5 .. 30 (26 values)
cache       : \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\_dataset\concat_source_v4


## 1 - Load the shared cohort

Built once by `rebuild_concat_cache.py`. This notebook only reads it.

In [2]:
# The cache MUST already exist. rebuild_concat_cache.py builds it once, serially.
# Building it here would mean three kernels racing to write the same files, and
# prepare_dataset's cache key carries no patient list, so a half-written cache would
# be picked up as a hit by whichever kernel arrived second.
assert CACHE_DIR.exists() and (CACHE_DIR / 'X_3d.npy').exists(), (
    f'{CACHE_DIR} missing - run:  python rebuild_concat_cache.py --apply')

df_contacts, X_concat = CC.build_concat_dataset(
    INPUT_DIR, conditions=('audio','picture','reading'),
    require_high_activity=True, cache_dir=CACHE_DIR, verbose=True)

X = {'concat_hg':    CC.concat_hg_features(X_concat, hg_band=(70.0,150.0), fmax=500.0),
     'concat_rawds': CC.concat_rawds_features(X_concat, n_blocks=3, fmax_hz=500.0)}

print(f'\ncohort: {len(df_contacts)} electrodes · '
      f'{df_contacts.patient_id.nunique()} patients')
for k_, v in X.items():
    print(f'  {k_:<14} {v.shape}')

[lf_dataset cache hit] \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\_dataset\concat_source_v4
  9342 samples · X_3d.shape=(9342, 129, 300)
[lf_concat] dropped 192 subdural GRID contacts {'PAT_3415': 192} — their depth contacts are kept
[lf_concat] excluded patients ['EL044'] — 124 rows removed
[lf_concat] 1693 electrodes (audio+picture+reading) · X_concat=(1693, 129, 900)
[lf_concat]   dropped 149 (missing a condition), 1266 (no high-activity condition)
[lf_concat]   high-activity in 1/2/3 conditions: 828 / 465 / 400
[build_hg_feature_matrix] X_hg.shape=(1693, 900)  hg_band=(70.0, 150.0) Hz  fmax=500.0 Hz

cohort: 1693 electrodes · 27 patients
  concat_hg      (1693, 900)
  concat_rawds   (1693, 1350)


## 2 - Fit Ward at every K

In [3]:
# one loop, both feature sets, every K in one fit_and_save call
results = {}
for fs in FEATURE_SETS:
    t0 = time.time()
    params = ({'k_range': K_RANGE, 'random_state': RANDOM_STATE, 'n_init': 20}
              if METHOD == 'kmeans' else
              {'linkage': 'ward', 'metric': 'euclidean', 'k_range': K_RANGE})
    m = R.fit_and_save(
        X[fs], df_keep=df_contacts, method=METHOD, feature_set=fs, params=params,
        method_label=METHOD_LBL,
        feature_set_label={'concat_hg':'Concatenated HG [a|p|r]',
                           'concat_rawds':'Concatenated 15-band [a|p|r]'}[fs],
        feature_names=CC.concat_feature_names(fs), notebook=SCRIPT_NAME)
    results[fs] = m
    print(f'{fs:<14} best_k={m["summary"]["best_k"]}   ({time.time()-t0:.0f}s)')

for fs, m in results.items():
    rd = LR.newest_run(METHOD, fs)
    lab = pd.read_csv(rd / 'cluster_labels_by_k.csv')
    have = sorted(int(c[2:]) for c in lab.columns if c.startswith('k_'))
    print(f'{fs:<14} {rd.name}   K columns {have[0]}..{have[-1]} ({len(have)})')
    assert set(K_RANGE).issubset(have), f'{fs}: missing K columns'

[HC] method=ward  metric=euclidean  n=1693  cophenetic_r=0.391
  HC sweep k=5: silhouette=0.132
  HC sweep k=6: silhouette=0.133
  HC sweep k=7: silhouette=0.058
  HC sweep k=8: silhouette=0.062
  HC sweep k=9: silhouette=0.057
  HC sweep k=10: silhouette=0.058
  HC sweep k=11: silhouette=0.061
  HC sweep k=12: silhouette=0.064
  HC sweep k=13: silhouette=0.059
  HC sweep k=14: silhouette=0.058
  HC sweep k=15: silhouette=0.059
  HC sweep k=16: silhouette=0.062
  HC sweep k=17: silhouette=0.058
  HC sweep k=18: silhouette=0.052
  HC sweep k=19: silhouette=0.056
  HC sweep k=20: silhouette=0.057
  HC sweep k=21: silhouette=0.058
  HC sweep k=22: silhouette=0.062
  HC sweep k=23: silhouette=0.061
  HC sweep k=24: silhouette=0.061
  HC sweep k=25: silhouette=0.063
  HC sweep k=26: silhouette=0.062
  HC sweep k=27: silhouette=0.064
  HC sweep k=28: silhouette=0.067
  HC sweep k=29: silhouette=0.069
  HC sweep k=30: silhouette=0.071
[fit_and_save] sweep statistics for K=[5, 6, 7, 8, 9, 10, 

## 3 - Held-out variance, K=5..30

Bi-cross-validation: a block of rows AND a block of columns is held out, so an extra component has to earn its place. Home space only - that is what the peak-K decision reads.

In [4]:
# Held-out variance, K=5..30, HOME SPACE only, bi-cross-validated.
# --tag keeps this kernel's output in its own file so the three notebooks running in
# parallel never write to the same CSV.
cmd = [sys.executable, 'make_heldout_variance.py',
       '--from-cache', str(CACHE_DIR),
       '--feature-set', *FEATURE_SETS,
       '--method', METHOD, '--tag', METHOD,
       '--ks', *[str(k) for k in K_RANGE],
       '--spaces', 'home', '--n-iter', '300']
print('$', ' '.join(cmd), flush=True)
r = subprocess.run(cmd, cwd=str(ROOT), env={**os.environ, 'PYTHONIOENCODING':'utf-8'})
assert r.returncode == 0

pk = pd.read_csv(ROOT/'outputs'/'clustering'/'bsf_comparison'/f'heldout_peaks_{METHOD}.csv')
display(pk)

$ C:\Users\artoni\.conda\envs\LORA_ENV2\python.exe make_heldout_variance.py --from-cache \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\_dataset\concat_source_v4 --feature-set concat_hg concat_rawds --method hierarchical --tag hierarchical --ks 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 --spaces home --n-iter 300


,feature_set,scheme,method_label,k_peak,peak,at_k8,monotone,k_max_tested
0,concat_hg,home,Ward,30,0.581937,0.449310,True,30
1,concat_rawds,home,Ward,30,0.441839,0.314129,True,30


## 4 - Done

In [5]:
print('DONE:', METHOD_LBL)
print('When ALL THREE of 240 / 241 / 242 have finished, run 243.')

DONE: Ward
When ALL THREE of 240 / 241 / 242 have finished, run 243.
